# Semana 11: Tareas Avanzadas de NLP: NER, Question Answering, Summarization
## Notebook de Ejercicios (NB2) – Fine-tuning para NER o QA

**Propósito:** Realizar fine-tuning de BERT para Named Entity Recognition (NER) usando el dataset CoNLL-2003, evaluar con métricas por entidad, y opcionalmente fine-tuner para Question Answering con SQuAD.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Fine-tunear BERT para NER usando Hugging Face.
- Evaluar el modelo con métricas por entidad (precisión, recall, F1).
- (Opcional) Fine-tunear BERT para QA y evaluar con exact match y F1.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalamos dependencias necesarias
!pip install transformers datasets evaluate seqeval accelerate

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
warnings.filterwarnings('ignore')

# Hugging Face
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_dataset, DatasetDict, ClassLabel
import evaluate

# Para seqeval (métrica NER)
import seqeval

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## PARTE 1: FINE-TUNING DE BERT PARA NER

### 1.1. Carga del Dataset CoNLL-2003

El dataset **CoNLL-2003** es un conjunto de datos estándar para reconocimiento de entidades nombradas en inglés.

In [ ]:
# Cargar dataset CoNLL-2003
print("Cargando dataset CoNLL-2003...")
raw_datasets = load_dataset("conll2003")

# Mostrar información del dataset
print(raw_datasets)

# Ver ejemplo
print("\nEjemplo de entrenamiento:")
idx = 0
print(f"Tokens: {raw_datasets['train'][idx]['tokens']}")
print(f"Etiquetas NER (etiquetas): {raw_datasets['train'][idx]['ner_tags']}")

# Obtener los nombres de las etiquetas
label_names = raw_datasets['train'].features['ner_tags'].feature.names
print(f"\nNombres de etiquetas: {label_names}")

# Mostrar etiquetas correspondientes al ejemplo
tags = [label_names[tag] for tag in raw_datasets['train'][idx]['ner_tags']]
print(f"Etiquetas (nombres): {tags}")

### 1.2. Tokenización y alineación de etiquetas

Para NER, necesitamos alinear las etiquetas con los sub-tokens generados por el tokenizador.

In [ ]:
# Cargar tokenizer de BERT
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Función de tokenización con alineación de etiquetas
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignorar tokens especiales
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # Para subpalabras, asignar -100 también (o etiqueta anterior según enfoque)
                label_ids.append(-100)  # Ignorar subpalabras
            previous_word_idx = word_idx
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Aplicar tokenización a los datasets
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("Tokenización completada.")
print(f"Columnas disponibles: {tokenized_datasets['train'].column_names}")

### 1.3. Carga del Modelo BERT para Token Classification

In [ ]:
# Cargar modelo con cabeza de clasificación por token
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id
)

# Verificar número de parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

### 1.4. Data Collator y Métricas

In [ ]:
# Data collator para padding dinámico
data_collator = DataCollatorForTokenClassification(tokenizer)

# Cargar métrica seqeval para NER
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Remover etiquetas ignoradas (-100)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### 1.5. Configuración del Entrenamiento

Usamos solo un subconjunto pequeño para que el entrenamiento sea rápido.

In [ ]:
# Crear subconjuntos pequeños para entrenamiento rápido
train_dataset = tokenized_datasets["train"].select(range(500))
eval_dataset = tokenized_datasets["validation"].select(range(100))
test_dataset = tokenized_datasets["test"].select(range(100))

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./bert-ner-conll",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

# Crear Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configurado.")

In [ ]:
# Entrenar modelo
print("Iniciando fine-tuning de BERT para NER...")
trainer.train()

### 1.6. Evaluación del Modelo NER

In [ ]:
# Evaluar en test
test_results = trainer.evaluate(test_dataset)
print("=== RESULTADOS NER EN TEST ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

### 1.7. Métricas por Entidad

Calculamos precisión, recall y F1 para cada tipo de entidad (PER, ORG, LOC, MISC).

In [ ]:
# Función para obtener métricas por entidad
def compute_entity_metrics(trainer, dataset):
    predictions, labels, _ = trainer.predict(dataset)
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    # Obtener métricas por entidad
    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    # Crear DataFrame
    entity_metrics = []
    for entity_type in ['PER', 'ORG', 'LOC', 'MISC']:
        if entity_type in results:
            entity_metrics.append({
                'Entidad': entity_type,
                'Precisión': results[entity_type]['precision'],
                'Recall': results[entity_type]['recall'],
                'F1': results[entity_type]['f1']
            })
    
    return pd.DataFrame(entity_metrics)

df_entity = compute_entity_metrics(trainer, test_dataset)
print("=== MÉTRICAS POR ENTIDAD ===")
df_entity.round(4)

In [ ]:
# Visualizar métricas por entidad
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Precisión', 'Recall', 'F1']
for i, metric in enumerate(metrics):
    axes[i].bar(df_entity['Entidad'], df_entity[metric])
    axes[i].set_title(f'{metric} por Entidad')
    axes[i].set_ylabel(metric)
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 1.8. Guardar el Modelo Fine-tuneado

In [ ]:
# Guardar modelo y tokenizer
model_save_path = "./bert-ner-conll-final"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Modelo guardado en {model_save_path}")

### 1.9. Prueba del Modelo NER con un Ejemplo Nuevo

In [ ]:
from transformers import pipeline

# Cargar pipeline de NER con nuestro modelo fine-tuneado
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Texto de ejemplo
texto_prueba = """
Elon Musk founded SpaceX in Hawthorne, California. Jeff Bezos started Amazon in Seattle.
"""

entities = ner_pipeline(texto_prueba)
print("=== ENTIDADES DETECTADAS ===")
for entity in entities:
    print(f"Entidad: {entity['word']:15} | Tipo: {entity['entity_group']:5} | Score: {entity['score']:.4f}")

---
## PARTE 2: FINE-TUNING PARA QUESTION ANSWERING (OPCIONAL)

Esta sección es opcional y requiere más recursos. Descomentar para ejecutar.

In [ ]:
# Descomentar para ejecutar QA
"""
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, DefaultDataCollator
from datasets import load_dataset
import evaluate

# Cargar dataset SQuAD
squad_dataset = load_dataset("squad")

# Cargar tokenizer y modelo para QA
model_checkpoint_qa = "bert-base-uncased"
tokenizer_qa = AutoTokenizer.from_pretrained(model_checkpoint_qa)
model_qa = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint_qa)

# Función de preprocesamiento para QA
def preprocess_qa(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer_qa(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
    
    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
        
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1
        
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)
    
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Tokenizar datasets (puede tomar tiempo)
tokenized_squad = squad_dataset.map(preprocess_qa, batched=True, remove_columns=squad_dataset["train"].column_names)

# Data collator
data_collator_qa = DefaultDataCollator()

# Métricas
metric_qa = evaluate.load("squad")

def compute_metrics_qa(eval_pred):
    predictions, references = eval_pred
    start_logits, end_logits = predictions
    # ... (implementación completa)
    pass

print("Configuración de QA completada. Para entrenar, descomentar el código completo.")
"""

---
## 8. Conclusiones

En este notebook hemos realizado fine-tuning de BERT para NER:

✔️ **Preprocesamiento**: Alineamos etiquetas NER con sub-tokens de BERT.

✔️ **Entrenamiento**: Fine-tuneamos BERT en CoNLL-2003 con el `Trainer` de Hugging Face.

✔️ **Evaluación por entidad**: Calculamos precisión, recall y F1 para cada tipo de entidad (PER, ORG, LOC, MISC).

✔️ **Resultados típicos**: F1 global > 0.90, con variaciones por entidad (LOC suele ser la más fácil, ORG la más difícil).

✔️ **QA opcional**: Se dejó código comentado para fine-tuning en SQuAD.

**Lección clave**: Fine-tunear modelos pre-entrenados para tareas específicas como NER es sencillo con Hugging Face y produce resultados de calidad con relativamente pocos datos.

---
**Fin del Notebook de Ejercicios - Semana 11 (NLP)**